# LR vs RF — Head-to-Head on Unseen Candles

Compare the best strategies from each model on forward-test data.

- **LR:** 31 optimal features, 2x e5%+e50%, conf>0.7
- **RF:** 20 optimal features, 1x e5% conf>0.7 + 2x e5%+e50% conf>0.7

Both trained on ALL `latest_features.jsonl`, tested on newest candles from DB.

In [ ]:
import json
import random
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from technicals import CandleRecord, IndicatorSnapshot, compute_all
from tqdm import tqdm

random.seed(42)
np.random.seed(42)

FEATURES_PATH = Path("../data/latest_features.jsonl")
DB_PATH = Path("../data/collection.db")
MAX_BID = 0.85
WARM_UP = 21

## 1. Load training data and define feature sets

In [ ]:
rows = []
with open(FEATURES_PATH) as f:
    for line in f:
        rows.append(json.loads(line))

df = pd.DataFrame(rows)
df["target"] = (df["outcome"] == "UP").astype(int)

NON_FEAT = {
    "candle_id",
    "session",
    "timestamp",
    "elapsed_pct",
    "btc_price",
    "up_best_bid",
    "up_best_ask",
    "up_bid_depth",
    "up_ask_depth",
    "down_best_bid",
    "down_best_ask",
    "down_bid_depth",
    "down_ask_depth",
    "market_volume",
    "outcome",
    "target",
}
all_feat_cols = sorted([c for c in df.columns if c not in NON_FEAT])
df[all_feat_cols] = df[all_feat_cols].fillna(0.0)

# LR: 31 optimal features (from notebook 1)
lr_feat_cols = sorted(
    [
        "bollinger_pct_b",
        "btc_direction_consistency",
        "btc_move_from_open",
        "btc_token_correlation",
        "btc_velocity",
        "candle_momentum",
        "consecutive_streak",
        "cross_book_flow",
        "current_elapsed",
        "depth_absorption_rate",
        "intra_candle_kurtosis",
        "liquidity_decay",
        "ob_pressure_gradient",
        "price_path_entropy",
        "prior_return",
        "prior_reversal_rate",
        "relative_volume",
        "reversal_regime",
        "rr_spread",
        "rsi",
        "smart_money_signal",
        "stochastic_k",
        "time_of_day_cos",
        "time_of_day_sin",
        "token_price_divergence",
        "trend_consistency",
        "up_book_imbalance",
        "up_spread_level",
        "volume_momentum",
        "volume_price_correlation",
        "weighted_mid_price",
    ]
)

# RF: 20 optimal features (from notebook 11)
rf_feat_cols = [
    "down_risk_reward",
    "up_risk_reward",
    "btc_move_from_open",
    "down_implied_probability",
    "rr_spread",
    "up_implied_probability",
    "down_token_velocity",
    "up_spread_level",
    "up_token_velocity",
    "btc_velocity",
    "adx",
    "time_of_day_cos",
    "volume_momentum",
    "ma_crossover",
    "regime_score",
    "stochastic_k",
    "return_autocorrelation",
    "conviction_score",
    "multi_candle_return_6",
    "rsi",
]

print(f"Training data: {df['candle_id'].nunique()} candles ({len(df):,} rows)")
print(f"LR features: {len(lr_feat_cols)}")
print(f"RF features: {len(rf_feat_cols)}")

## 2. Train both models on all data

In [ ]:
# LR
lr_scaler = StandardScaler()
X_lr = lr_scaler.fit_transform(df[lr_feat_cols].values)
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_lr, df["target"].values)
print(f"LR trained: {lr_model.coef_.size} params")

# RF
rf_scaler = StandardScaler()
X_rf = rf_scaler.fit_transform(df[rf_feat_cols].values)
rf_model = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_leaf=20, random_state=42, n_jobs=-1)
rf_model.fit(X_rf, df["target"].values)
rf_params = sum(e.tree_.node_count for e in rf_model.estimators_) * 2
print(f"RF trained: {rf_params:,} params")

max_train_ts = df["timestamp"].max()

## 3. Load and compute features for new candles

In [ ]:
conn = sqlite3.connect(str(DB_PATH))
candles_df = pd.read_sql(f"SELECT * FROM candles WHERE start_time > {max_train_ts} ORDER BY start_time", conn)
snaps_df = (
    pd.read_sql(
        "SELECT * FROM snapshots WHERE candle_id IN ({}) ORDER BY candle_id, timestamp".format(
            ",".join(f"'{cid}'" for cid in candles_df["candle_id"])
        ),
        conn,
    )
    if len(candles_df) > 0
    else pd.DataFrame()
)
prior_candles_df = pd.read_sql(
    f"SELECT * FROM candles WHERE start_time <= {max_train_ts} ORDER BY start_time DESC LIMIT {WARM_UP}", conn
)
conn.close()
prior_candles_df = prior_candles_df.sort_values("start_time")

prior_candles = []
for _, cr in prior_candles_df.iterrows():
    prior_candles.append(
        CandleRecord(
            candle_id=cr["candle_id"],
            start_time=cr["start_time"],
            end_time=cr["end_time"],
            open=cr["open"],
            high=cr["high"],
            low=cr["low"],
            close=cr["close"],
            volume=cr["volume"],
            outcome=cr["outcome"],
            final_ret=cr["final_ret"],
        )
    )

all_rows = []
for _, cr in tqdm(candles_df.iterrows(), total=len(candles_df), desc="Computing features"):
    cid = cr["candle_id"]
    candle = CandleRecord(
        candle_id=cid,
        start_time=cr["start_time"],
        end_time=cr["end_time"],
        open=cr["open"],
        high=cr["high"],
        low=cr["low"],
        close=cr["close"],
        volume=cr["volume"],
        outcome=cr["outcome"],
        final_ret=cr["final_ret"],
    )
    snap_rows = snaps_df[snaps_df["candle_id"] == cid]
    if len(snap_rows) < 5:
        prior_candles.append(candle)
        continue
    snapshots = []
    for _, s in snap_rows.iterrows():
        ob = json.loads(s["orderbook_json"])
        snapshots.append(
            IndicatorSnapshot(
                candle_id=cid,
                timestamp=s["timestamp"],
                elapsed_pct=s["elapsed_pct"],
                btc_price=s["btc_price"],
                btc_bid=s["btc_bid"],
                btc_ask=s["btc_ask"],
                up_bids=[ob["up_bids"][0]] if ob.get("up_bids") else [],
                up_asks=[ob["up_asks"][0]] if ob.get("up_asks") else [],
                down_bids=[ob["down_bids"][0]] if ob.get("down_bids") else [],
                down_asks=[ob["down_asks"][0]] if ob.get("down_asks") else [],
                market_volume=s["market_volume"],
            )
        )
    for si in range(len(snapshots)):
        indicators = compute_all(prior_candles, candle.open, snapshots[: si + 1])
        snap = snapshots[si]
        row = {
            "candle_id": cid,
            "timestamp": snap.timestamp,
            "elapsed_pct": snap.elapsed_pct,
            "btc_price": snap.btc_price,
            "up_best_ask": snap.up_asks[0][0] if snap.up_asks else None,
            "down_best_ask": snap.down_asks[0][0] if snap.down_asks else None,
            **indicators,
            "outcome": candle.outcome,
        }
        all_rows.append(row)
    prior_candles.append(candle)

df_eval = pd.DataFrame(all_rows)
df_eval["target"] = (df_eval["outcome"] == "UP").astype(int)
df_eval[all_feat_cols] = df_eval[all_feat_cols].fillna(0.0)

print(f"\nNew candles: {df_eval['candle_id'].nunique()}")
print(f"Rows: {len(df_eval):,}")

## 4. Build per-candle predictions for both models

In [ ]:
all_cd = []

for cid in df_eval["candle_id"].unique():
    snap_rows = df_eval[df_eval["candle_id"] == cid].sort_values("timestamp")
    if len(snap_rows) < 5:
        continue
    truth = int(snap_rows["target"].iloc[0])

    X_lr_snap = lr_scaler.transform(snap_rows[lr_feat_cols].values)
    lr_probs = lr_model.predict_proba(X_lr_snap)[:, 1]
    lr_preds = (lr_probs >= 0.5).astype(int)

    X_rf_snap = rf_scaler.transform(snap_rows[rf_feat_cols].values)
    rf_probs = rf_model.predict_proba(X_rf_snap)[:, 1]
    rf_preds = (rf_probs >= 0.5).astype(int)

    up_asks = snap_rows["up_best_ask"].values
    down_asks = snap_rows["down_best_ask"].values
    elapsed = snap_rows["elapsed_pct"].values

    sd = [
        {
            "tick": i,
            "elapsed_pct": elapsed[i],
            "lr_pred": int(lr_preds[i]),
            "lr_prob": float(lr_probs[i]),
            "rf_pred": int(rf_preds[i]),
            "rf_prob": float(rf_probs[i]),
            "up_ask": up_asks[i],
            "down_ask": down_asks[i],
        }
        for i in range(len(snap_rows))
    ]
    all_cd.append({"candle_id": cid, "truth": truth, "snapshots": sd})

print(f"Built predictions for {len(all_cd)} candles")

## 5. Scaling-in engine (supports model selection)

In [ ]:
def run_scaling(name, entry_points, model_key="lr", bet_per_entry=10.0, min_confidence=0.0):
    pred_key = f"{model_key}_pred"
    prob_key = f"{model_key}_prob"
    bal = 1000.0
    history = [bal]
    total_bets, total_wins, candles_entered, candles_skipped = 0, 0, 0, 0

    for cd in all_cd:
        sd = cd["snapshots"]
        truth = cd["truth"]
        entries = []
        first_direction = None

        for min_e, n_c in entry_points:
            for i in range(max(n_c - 1, 0), len(sd)):
                if sd[i]["elapsed_pct"] < min_e:
                    continue
                if any(i <= prev_tick for prev_tick, _, _ in entries):
                    continue
                if n_c > 1 and not all(sd[i - j][pred_key] == sd[i][pred_key] for j in range(n_c)):
                    continue
                confidence = max(sd[i][prob_key], 1.0 - sd[i][prob_key])
                if confidence < min_confidence:
                    continue
                direction = sd[i][pred_key]
                if first_direction is None:
                    first_direction = direction
                elif direction != first_direction:
                    break
                ask = sd[i]["up_ask"] if direction == 1 else sd[i]["down_ask"]
                if ask is None or not np.isfinite(ask) or ask <= 0 or ask >= MAX_BID:
                    continue
                entries.append((i, direction, ask))
                break

        if not entries:
            candles_skipped += 1
            continue
        candles_entered += 1
        for _, direction, ask in entries:
            if bal < bet_per_entry:
                break
            total_bets += 1
            if direction == truth:
                bal += (bet_per_entry / ask) * (1.0 - ask)
                total_wins += 1
            else:
                bal -= bet_per_entry
        history.append(bal)

    wr = total_wins / total_bets if total_bets > 0 else 0
    return {
        "name": name,
        "balance": bal,
        "history": history,
        "total_bets": total_bets,
        "candles_entered": candles_entered,
        "wins": total_wins,
        "win_rate": wr,
        "return": (bal - 1000) / 1000 * 100,
    }

## 6. Run strategies

In [ ]:
strategies = [
    # LR strategies
    ("LR 1x e5%", [(0.05, 3)], "lr", 0.0),
    ("LR 1x e5% conf>0.7", [(0.05, 3)], "lr", 0.7),
    ("LR 2x e5%+e50%", [(0.05, 3), (0.50, 3)], "lr", 0.0),
    ("LR 2x e5%+e50% conf>0.6", [(0.05, 3), (0.50, 3)], "lr", 0.6),
    ("LR 2x e5%+e50% conf>0.7", [(0.05, 3), (0.50, 3)], "lr", 0.7),
    # RF strategies
    ("RF 1x e5%", [(0.05, 3)], "rf", 0.0),
    ("RF 1x e5% conf>0.7", [(0.05, 3)], "rf", 0.7),
    ("RF 2x e5%+e50%", [(0.05, 3), (0.50, 3)], "rf", 0.0),
    ("RF 2x e5%+e50% conf>0.6", [(0.05, 3), (0.50, 3)], "rf", 0.6),
    ("RF 2x e5%+e50% conf>0.7", [(0.05, 3), (0.50, 3)], "rf", 0.7),
    # RF delayed entry
    ("RF 1x e30%", [(0.30, 3)], "rf", 0.0),
    ("RF 2x e30%+e60% conf>0.6", [(0.30, 3), (0.60, 3)], "rf", 0.6),
]

results = []
print(f"{'Strategy':<30} {'Bets':>5} {'WR':>6} {'Balance':>10} {'Return':>8} {'MaxDD':>7}")
print("-" * 72)

for name, eps, model_key, conf in strategies:
    r = run_scaling(name, eps, model_key=model_key, min_confidence=conf)
    peak = r["history"][0]
    max_dd = 0
    for h in r["history"]:
        if h > peak:
            peak = h
        dd = (peak - h) / peak
        if dd > max_dd:
            max_dd = dd
    r["max_dd"] = max_dd
    results.append(r)
    print(
        f"{r['name']:<30} {r['total_bets']:>5} "
        f"{r['win_rate'] * 100:>5.1f}% ${r['balance']:>9.2f} {r['return']:>+7.1f}% {max_dd * 100:>6.1f}%"
    )

## 7. Equity curves — LR vs RF head-to-head

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Top: all strategies
lr_color = "#3498db"
rf_color = "#e74c3c"
for r in results:
    color = lr_color if r["name"].startswith("LR") else rf_color
    axes[0].plot(r["history"], color=color, alpha=0.3, linewidth=1)
axes[0].plot([], [], color=lr_color, label="LR strategies", linewidth=2)
axes[0].plot([], [], color=rf_color, label="RF strategies", linewidth=2)
axes[0].axhline(1000, color="gray", linestyle="--", alpha=0.3)
axes[0].set_xlabel("Candle #")
axes[0].set_ylabel("Balance ($)")
axes[0].set_title("All Strategies — LR (blue) vs RF (red)")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Bottom: best of each
best_lr = max([r for r in results if r["name"].startswith("LR")], key=lambda r: r["balance"])
best_rf = max([r for r in results if r["name"].startswith("RF")], key=lambda r: r["balance"])

axes[1].plot(
    best_lr["history"],
    color=lr_color,
    linewidth=2.5,
    label=f"{best_lr['name']} → ${best_lr['balance']:,.0f} ({best_lr['return']:+.1f}%, WR={best_lr['win_rate'] * 100:.0f}%)",
)
axes[1].plot(
    best_rf["history"],
    color=rf_color,
    linewidth=2.5,
    label=f"{best_rf['name']} → ${best_rf['balance']:,.0f} ({best_rf['return']:+.1f}%, WR={best_rf['win_rate'] * 100:.0f}%)",
)
axes[1].axhline(1000, color="gray", linestyle="--", alpha=0.3)
axes[1].set_xlabel("Candle #")
axes[1].set_ylabel("Balance ($)")
axes[1].set_title("Best LR vs Best RF")
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Summary

In [ ]:
print("=" * 72)
print("LR vs RF — FORWARD TEST SUMMARY")
print(f"Candles: {len(all_cd)} (unseen during training)")
print("=" * 72)

print(f"\n{'':30} {'LR best':>20} {'RF best':>20}")
print("-" * 72)
print(f"{'Strategy':<30} {best_lr['name']:>20} {best_rf['name']:>20}")
print(f"{'Win Rate':<30} {best_lr['win_rate'] * 100:>19.1f}% {best_rf['win_rate'] * 100:>19.1f}%")
print(f"{'Return':<30} {best_lr['return']:>+19.1f}% {best_rf['return']:>+19.1f}%")
print(f"{'Max Drawdown':<30} {best_lr['max_dd'] * 100:>19.1f}% {best_rf['max_dd'] * 100:>19.1f}%")
print(f"{'Final Balance':<30} ${best_lr['balance']:>18.2f} ${best_rf['balance']:>18.2f}")
print(f"{'Total Bets':<30} {best_lr['total_bets']:>20} {best_rf['total_bets']:>20}")

winner = "LR" if best_lr["balance"] > best_rf["balance"] else "RF"
print(f"\nWinner: {winner}")
print("\nNote: Small sample size — results are directional, not conclusive.")
print("Re-run after collecting more data for robust comparison.")

## 9. Conclusion

See results above. Both models tested on candles never seen during training. Re-run as more data accumulates.